In [19]:
sql_query = """
WITH Ranked AS
(
    SELECT
        schema_name,
        object_name,
        object_type,
        object_definition,
        capture_datetime,
        ROW_NUMBER() OVER
        (
            PARTITION BY object_name
            ORDER BY capture_datetime DESC
        ) AS rn
    FROM dbo.viewdefinitionaudit
    WHERE CAST(capture_datetime AS date) = CAST(GETDATE() AS date)
    and schema_name = 'dbo'
)

SELECT
    cur.object_name,
    cur.schema_name,
    cur.capture_datetime AS current_capture,
    prev.capture_datetime AS previous_capture,
    cur.object_definition AS view_v2,
    prev.object_definition AS view_v1
FROM Ranked cur
OUTER APPLY
(
    SELECT TOP (1)
        capture_datetime,
        object_definition
    FROM dbo.viewdefinitionaudit p
    WHERE
        p.object_name = cur.object_name
        AND p.capture_datetime < cur.capture_datetime
    ORDER BY p.capture_datetime DESC
) prev
WHERE
    cur.rn = 1
    AND prev.object_definition IS NOT NULL;
"""



In [ ]:
import pyodbc
import os
import csv
import pandas as pd
from gpt4all import GPT4All
import difflib
import pandas as pd
from IPython.display import HTML

DriverName = "ODBC Driver 17 for SQL Server"
ServerName = "datawarehouse.fabric.microsoft.com"
DatabaseName = "dev_database"
UID = "nazeer.joseph@email.com"

conn = pyodbc.connect(
    f'DRIVER={{{DriverName}}};'
    f'SERVER={ServerName};'
    f'DATABASE={DatabaseName};'
    f'UID={UID};'
    'Authentication=ActiveDirectoryInteractive;'
)

In [21]:

# Load model once
model = GPT4All(
    "qwen2.5-coder-7b-instruct-q4_0.gguf",
    n_ctx=8192
)


def summarize_view_changes(view_v1, view_v2):
    diff_lines = [
        line
        for line in difflib.unified_diff(
            view_v1.splitlines(),
            view_v2.splitlines(),
            lineterm=""
        )
        if (line.startswith("+") or line.startswith("-"))
        and not line.startswith("+++")
        and not line.startswith("---")
    ]

    if not diff_lines:
        return "- No logical changes."

    diff = "\n".join(diff_lines)

    prompt = f"""
You are a senior SQL code reviewer.

Below is a unified diff of a SQL VIEW.
Lines beginning with '-' were removed.
Lines beginning with '+' were added.

Describe every meaningful logical change.

Rules:
- One bullet per logical change.
- Maximum 15 words per bullet.
- State what changed FROM and TO where applicable.
- Describe the business effect rather than SQL syntax.
- If a field, calculation, filter or mapping changed, mention both the old and new behaviour.
- Group related code changes into one bullet.
- Ignore formatting, whitespace, schema prefixes and SQL boilerplate.
- Infer business meaning from aliases and expressions.
- Do not mention SQL objects or implementation details.
- If there are no logical changes, output:
  - No logical changes.
- Output only the bullets.

{diff}
"""

    with model.chat_session():
        return model.generate(
            prompt=prompt,
            temp=0,
            max_tokens=200
        ).strip()


# Read SQL results
df = pd.read_sql(sql_query, conn)

df["Change"] = df.apply(
    lambda row: summarize_view_changes(
        row["view_v1"],
        row["view_v2"]
    ),
    axis=1
)

df = df[
    [
        "object_name",
        "Change",
        "current_capture",
        "previous_capture"
    ]
]

display(HTML(df.to_html(index=False)))

conn.close()



C:\Users\nazeer.joseph\AppData\Local\Temp\ipykernel_29436\1231730137.py:61: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql_query, conn)


object_name,Change,current_capture,previous_capture
ns2_transactions_vw,- Changed invoiced project identifier from custbody\_kbs\_shellproject to custbody\_kbs\_chargeinvoiceproject.,2026-06-26 05:02:37.406874,2026-06-23 12:46:30.235205
